# Encoding Texts & Material Data — Exercices pratiques
**Computational Philology — Séance 3**

---

Ce carnet est structuré en **trois parties progressives** :

| Partie | Contenu | Durée estimée |
|--------|---------|---------------|
| **1. XML TEI** | Encodage d'un apparat critique (NT grec, Jean 1:1-5) | ~20 min |
| **2. XSLT via Python** | Transformation TEI → HTML (Vetus Latina, Genèse 1:1-3) | ~10 min |
| **3. JSON** | Modélisation et requêtes (conversion et interrogation) | ~15 min |

> **Note technique** : ce carnet utilise uniquement `lxml` (préinstallé sur JupyterHub et Google Colab). Aucune installation supplémentaire n'est nécessaire.

In [ ]:
# Cellule d'initialisation — exécutez-la en premier
from lxml import etree
import json
from IPython.display import HTML, display
print("✓ Bibliothèques chargées")

---
## Partie 1 — XML TEI : Nouveau Testament grec (Jean 1:1-5)

### Contexte philologique

Le texte grec du Nouveau Testament nous est transmis par plus de **5 000 manuscrits grecs**. Les principaux témoins pour l'Évangile de Jean sont :

| Sigle | Témoin | Date | Lieu de conservation |
|-------|--------|------|----------------------|
| **𝔓⁶⁶** | Papyrus Bodmer II | c. 200 | Bibliotheca Bodmeriana, Genève |
| **א** (Sinaïticus) | Codex Sinaiticus | c. 330-360 | British Library, Londres |
| **B** (Vaticanus) | Codex Vaticanus | c. 300-325 | Bibliothèque apostolique vaticane |
| **D** (Bezae) | Codex Bezae | c. 400-450 | Cambridge, Corpus Christi |

Le texte de référence utilisé est le **Nestle-Aland 28** (NA28).

### Jean 1:1-3 — Variantes principales

```
v.1 : Ἐν ἀρχῇ ἦν ὁ λόγος, καὶ ὁ λόγος ἦν πρὸς τὸν θεόν, καὶ θεὸς ἦν ὁ λόγος.
v.2 : οὗτος ἦν ἐν ἀρχῇ πρὸς τὸν θεόν.
v.3 : πάντα δι᾽ αὐτοῦ ἐγένετο, καὶ χωρὶς αὐτοῦ ἐγένετο οὐδὲ ἕν.
```

Variante notable au v.3/4 (ponctuation) :
- **א, B, 𝔓⁶⁶** : `…οὐδὲ ἕν. ὃ γέγονεν` (coupure avant *ὃ γέγονεν*)
- **D, versions latines** : `…οὐδὲ ἕν ὃ γέγονεν.` (coupure après *ὃ γέγονεν*)

Cette différence de ponctuation a des **implications théologiques majeures** (elle change qui « vivifie »).

---

### 1.1 — Un document TEI minimal vous est fourni

Lisez attentivement ce document, puis répondez aux questions en dessous.

In [ ]:
# Document TEI fourni — Jean 1:1-2 avec apparat
# Lisez-le attentivement avant de passer aux exercices

tei_fourni = """
<?xml version="1.0" encoding="UTF-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">

  <teiHeader>
    <fileDesc>
      <titleStmt>
        <title>Evangelium secundum Iohannem — Édition critique (extrait)</title>
      </titleStmt>
      <publicationStmt>
        <publisher>Cours de Philologie Computationnelle</publisher>
      </publicationStmt>
      <sourceDesc>
        <listWit>
          <witness xml:id="P66">Papyrus Bodmer II (𝔓⁶⁶), c. 200</witness>
          <witness xml:id="Sin">Codex Sinaiticus (א), c. 350</witness>
          <witness xml:id="Vat">Codex Vaticanus (B), c. 325</witness>
          <witness xml:id="Bez">Codex Bezae (D), c. 450</witness>
        </listWit>
      </sourceDesc>
    </fileDesc>
  </teiHeader>

  <text xml:lang="grc">
    <body>
      <div type="book" n="John">
        <div type="chapter" n="1">

          <ab n="1">
            Ἐν ἀρχῇ ἦν ὁ λόγος, καὶ ὁ λόγος ἦν πρὸς τὸν θεόν,
            καὶ
            <app>
              <lem wit="#P66 #Sin #Vat">θεὸς ἦν ὁ λόγος</lem>
              <rdg wit="#Bez" type="addition">ὁ θεὸς ἦν ὁ λόγος</rdg>
            </app>
            .
          </ab>

          <ab n="2">
            οὗτος ἦν ἐν ἀρχῇ πρὸς τὸν θεόν.
          </ab>

        </div>
      </div>
    </body>
  </text>

</TEI>
"""

# Validation de bonne formation XML
try:
    doc = etree.fromstring(tei_fourni.encode('utf-8'))
    print("✓ Document XML bien formé")
    print(f"  Élément racine : <{doc.tag.split('}')[1]}>")
except etree.XMLSyntaxError as e:
    print(f"✗ Erreur XML : {e}")

### 1.2 — Interroger le document avec lxml

La librairie `lxml` permet d'interroger un document XML avec **XPath**. 
Notez la gestion du namespace TEI (`http://www.tei-c.org/ns/1.0`) — c'est un piège classique.

In [ ]:
# Namespace TEI — indispensable pour les requêtes XPath
ns = {'tei': 'http://www.tei-c.org/ns/1.0'}

# Exemple 1 : lister les témoins déclarés
temoins = doc.findall('.//tei:witness', ns)
print("=== Témoins déclarés ===")
for t in temoins:
    print(f"  [{t.get('{http://www.w3.org/XML/1998/namespace}id')}] {t.text.strip()}")

print()

# Exemple 2 : extraire tous les apparat (app)
apparat = doc.findall('.//tei:app', ns)
print(f"=== Nombre d'unités d'apparat : {len(apparat)} ===")
for i, app in enumerate(apparat, 1):
    lem = app.find('tei:lem', ns)
    rdgs = app.findall('tei:rdg', ns)
    print(f"\n  App {i} :")
    print(f"    Lemma  : {lem.text.strip()} — témoins : {lem.get('wit')}")
    for r in rdgs:
        print(f"    Variante : {r.text.strip()} — témoins : {r.get('wit')} — type : {r.get('type')}")

---
### 1.3 — ✏️ Exercice : encoder Jean 1:3-4 avec l'apparat de ponctuation

Vous devez encoder le verset 3 avec la **variante de ponctuation** décrite dans l'introduction :

- **Lemma** (א, B, 𝔓⁶⁶) : `πάντα δι᾽ αὐτοῦ ἐγένετο, καὶ χωρὶς αὐτοῦ ἐγένετο οὐδὲ ἕν.` 
  → La phrase s'arrête là. *ὃ γέγονεν* commence le verset 4.
- **Variante** (D + versions latines) : la coupure se fait *après* `ὃ γέγονεν` 
  → `…οὐδὲ ἕν ὃ γέγονεν.` constitue une unité.

**Indices :**
- Utilisez `<app>`, `<lem wit="...">`, `<rdg wit="..." type="ponctuation">`
- La variante concerne la **coupure syntaxique** : vous pouvez encoder `ὃ γέγονεν` dans le `<rdg>` pour montrer qu'il est attaché différemment
- Référencez les témoins avec `#P66`, `#Sin`, `#Vat`, `#Bez`

> 💡 **Pour les plus avancés** : ajoutez aussi un élément `<note>` à l'intérieur de l'`<app>` pour expliquer l'enjeu théologique de cette variante.

In [ ]:
# ✏️ VOTRE RÉPONSE — complétez le TEI ci-dessous
# Remplacez les ... par votre encodage

tei_exercice_1 = """
<?xml version="1.0" encoding="UTF-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">

  <teiHeader>
    <fileDesc>
      <titleStmt>
        <title>Jean 1:3-4 — Exercice d'encodage</title>
      </titleStmt>
      <publicationStmt><publisher>Exercice</publisher></publicationStmt>
      <sourceDesc>
        <listWit>
          <witness xml:id="P66">Papyrus Bodmer II (𝔓⁶⁶)</witness>
          <witness xml:id="Sin">Codex Sinaiticus (א)</witness>
          <witness xml:id="Vat">Codex Vaticanus (B)</witness>
          <witness xml:id="Bez">Codex Bezae (D)</witness>
        </listWit>
      </sourceDesc>
    </fileDesc>
  </teiHeader>

  <text xml:lang="grc">
    <body>
      <div type="book" n="John">
        <div type="chapter" n="1">

          <ab n="3">
            πάντα δι᾽ αὐτοῦ ἐγένετο, καὶ χωρὶς αὐτοῦ ἐγένετο
            <app>
              <!-- VOTRE ENCODAGE ICI -->
              ...
            </app>
          </ab>

        </div>
      </div>
    </body>
  </text>

</TEI>
"""

# Testez la bonne formation de votre XML :
try:
    doc_ex1 = etree.fromstring(tei_exercice_1.encode('utf-8'))
    print("✓ XML bien formé !")
    # Vérification automatique : y a-t-il bien un <app> avec <lem> et <rdg> ?
    ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
    apps = doc_ex1.findall('.//tei:app', ns)
    if apps:
        for app in apps:
            lem = app.find('tei:lem', ns)
            rdgs = app.findall('tei:rdg', ns)
            print(f"  ✓ <app> trouvé avec {len(rdgs)} variante(s)")
            if lem is not None:
                print(f"    Lemma : '{lem.text.strip() if lem.text else '(vide)'}' — wit={lem.get('wit')}")
            for r in rdgs:
                print(f"    Variante : '{r.text.strip() if r.text else '(vide)'}' — wit={r.get('wit')}")
    else:
        print("  ⚠ Aucun élément <app> trouvé — avez-vous complété l'encodage ?")
except etree.XMLSyntaxError as e:
    print(f"✗ Erreur XML : {e}")
    print("  → Vérifiez que toutes vos balises sont bien fermées et imbriquées.")

<details>
<summary>💡 Solution (cliquez pour révéler)</summary>

```xml
<ab n="3">
  πάντα δι᾽ αὐτοῦ ἐγένετο, καὶ χωρὶς αὐτοῦ ἐγένετο
  <app>
    <lem wit="#P66 #Sin #Vat">οὐδὲ ἕν.</lem>
    <rdg wit="#Bez" type="ponctuation">οὐδὲ ἕν ὃ γέγονεν.</rdg>
    <note>La coupure syntaxique déplace ὃ γέγονεν du v.4 au v.3,
    modifiant radicalement le sujet de ζωή : dans la leçon de D,
    c'est la création qui est associée à la vie, non le Logos.</note>
  </app>
</ab>
```
</details>

---
### 1.4 — ✏️ Exercice avancé : encoder des phénomènes de transmission

Le Codex Bezae (D) est célèbre pour ses **corrections** et **ajouts marginaux**.
Encodez le verset 1 en ajoutant :

1. Une **correction** (le copiste a d'abord écrit `θεός` sans article, puis l'a corrigé) → `<del>` / `<add>`
2. Une **lecture incertaine** sur `λόγος` (encre pâlie) → `<unclear reason="faded">`
3. Une **abréviation** développée : `θς` pour `θεός` (nomen sacrum) → `<choice><abbr>/<expan>`

> Ces éléments vont dans la description du témoin D spécifiquement — vous pouvez créer un `<div>` dédié avec `@source="#Bez"`.

In [ ]:
# ✏️ VOTRE RÉPONSE
tei_exercice_1b = """
<?xml version="1.0" encoding="UTF-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">
  <teiHeader>
    <fileDesc>
      <titleStmt><title>Jean 1:1 — Transcription Codex Bezae</title></titleStmt>
      <publicationStmt><publisher>Exercice</publisher></publicationStmt>
      <sourceDesc><p>Codex Bezae Cantabrigiensis</p></sourceDesc>
    </fileDesc>
  </teiHeader>
  <text xml:lang="grc">
    <body>
      <div type="transcription" source="#Bez">
        <ab n="1">
          <!-- VOTRE ENCODAGE ICI -->
          ...
        </ab>
      </div>
    </body>
  </text>
</TEI>
"""

try:
    doc_ex1b = etree.fromstring(tei_exercice_1b.encode('utf-8'))
    print("✓ XML bien formé")
    ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
    for tag in ['unclear', 'del', 'add', 'choice']:
        found = doc_ex1b.findall(f'.//tei:{tag}', ns)
        status = '✓' if found else '·'
        print(f"  {status} <{tag}> : {len(found)} occurrence(s)")
except etree.XMLSyntaxError as e:
    print(f"✗ Erreur XML : {e}")

---
## Partie 2 — XSLT : Vetus Latina (Genèse 1:1-3)

### Contexte philologique

La **Vetus Latina** (ou *Vetus Itala*) désigne les traductions latines de la Bible antérieures à la Vulgate de Jérôme (fin IVe s.). Elle n'est pas un texte unique mais une **famille de traductions** présentant de nombreuses divergences entre elles et avec la Vulgate.

| Sigle | Source | Date approx. |
|-------|--------|--------------|
| **VL** | Vetus Latina (ms. Lugdunensis) | IIe-IVe s. |
| **Vulg** | Vulgate (Jérôme) | 382-405 |
| **Aug** | Augustin, *De Genesi ad litteram* | c. 401 |

**Genèse 1:1-2 — variantes :**

| Verset | Vetus Latina | Vulgate |
|--------|-------------|--------|
| 1:1 | *In principio fecit Deus caelum et terram* | *In principio creavit Deus caelum et terram* |
| 1:2 | *terra autem erat inanis et vacua* | *terra autem erat inanis et vacua* (identique) |

La variante principale est **`fecit` vs `creavit`** — un choix lexical qui a des implications pour la théologie de la création.

### 2.1 — Document TEI fourni

In [ ]:
# TEI de la Vetus Latina / Vulgate — Genèse 1:1-3

tei_genesis = """
<?xml version="1.0" encoding="UTF-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">

  <teiHeader>
    <fileDesc>
      <titleStmt>
        <title>Genesis 1:1-3 — Collation Vetus Latina / Vulgate</title>
      </titleStmt>
      <publicationStmt><publisher>Cours de Philologie Computationnelle</publisher></publicationStmt>
      <sourceDesc>
        <listWit>
          <witness xml:id="VL">Vetus Latina (Codex Lugdunensis, VIe s.)</witness>
          <witness xml:id="Vulg">Vulgata Hieronymi (c. 400)</witness>
          <witness xml:id="Aug">Augustinus, De Genesi ad litteram (c. 401)</witness>
        </listWit>
      </sourceDesc>
    </fileDesc>
  </teiHeader>

  <text xml:lang="la">
    <body>
      <div type="book" n="Genesis">
        <div type="chapter" n="1">

          <ab n="1">
            In principio
            <app>
              <lem wit="#Vulg">creavit</lem>
              <rdg wit="#VL" type="lexical">fecit</rdg>
              <rdg wit="#Aug" type="lexical">fecit</rdg>
            </app>
            Deus caelum et terram.
          </ab>

          <ab n="2">
            Terra autem erat inanis et vacua, et tenebrae erant super faciem
            <app>
              <lem wit="#Vulg #VL">abyssi</lem>
              <rdg wit="#Aug" type="orthographic">abissi</rdg>
            </app>
            , et spiritus
            <app>
              <lem wit="#Vulg">Dei</lem>
              <rdg wit="#VL" type="lexical">Domini</rdg>
              <rdg wit="#Aug" type="lexical">Dei</rdg>
            </app>
            ferebatur super aquas.
          </ab>

          <ab n="3">
            Dixitque Deus : fiat lux. Et
            <app>
              <lem wit="#Vulg #VL #Aug">facta est lux</lem>
            </app>
            .
          </ab>

        </div>
      </div>
    </body>
  </text>

</TEI>
"""

doc_gen = etree.fromstring(tei_genesis.encode('utf-8'))
print("✓ Document Genèse chargé")

### 2.2 — Transformation XSLT : produire un affichage HTML de l'apparat

Le XSLT ci-dessous vous est **fourni** : il transforme le TEI en une page HTML avec :
- le texte de base affiché normalement
- les variantes affichées en note de bas de page numérotée

**Votre tâche** : lire le XSLT, comprendre sa logique, puis l'exécuter.

In [ ]:
# XSLT fourni — transformation TEI → HTML avec apparat en notes
# Lisez-le avant de l'exécuter !

xslt_apparat = """
<?xml version="1.0" encoding="UTF-8"?>
<xsl:stylesheet version="1.0"
  xmlns:xsl="http://www.w3.org/1999/XSL/Transform"
  xmlns:tei="http://www.tei-c.org/ns/1.0">

  <xsl:output method="html" encoding="UTF-8" indent="yes"/>

  <!-- Racine -->
  <xsl:template match="/">
    <html>
      <head>
        <meta charset="UTF-8"/>
        <style>
          body { font-family: Georgia, serif; max-width: 750px;
                 margin: 2em auto; line-height: 1.8; color: #222; }
          h1   { font-size: 1.3em; color: #555; border-bottom: 1px solid #ccc;
                 padding-bottom: 0.3em; }
          .verse { margin: 0.5em 0; }
          .verse-num { font-weight: bold; color: #8B0000;
                       margin-right: 0.4em; }
          .app-marker { color: #8B0000; font-size: 0.75em;
                        vertical-align: super; cursor: help; }
          .apparatus { margin-top: 2em; border-top: 2px solid #ccc;
                       padding-top: 1em; font-size: 0.85em; }
          .app-entry { margin: 0.3em 0; }
          .sigle     { font-weight: bold; color: #1a5276; }
          .rdg-type  { color: #888; font-style: italic; font-size: 0.9em; }
        </style>
      </head>
      <body>
        <h1><xsl:value-of select="//tei:titleStmt/tei:title"/></h1>
        <xsl:apply-templates select="//tei:body"/>
        <div class="apparatus">
          <strong>Apparatus criticus</strong>
          <xsl:apply-templates select="//tei:app" mode="apparatus"/>
        </div>
      </body>
    </html>
  </xsl:template>

  <!-- Versets -->
  <xsl:template match="tei:ab">
    <div class="verse">
      <span class="verse-num"><xsl:value-of select="@n"/></span>
      <xsl:apply-templates/>
    </div>
  </xsl:template>

  <!-- Unité d'apparat dans le texte : affiche le lemma + marqueur -->
  <xsl:template match="tei:app">
    <xsl:variable name="pos">
      <xsl:number count="tei:app" level="any"/>
    </xsl:variable>
    <xsl:value-of select="tei:lem"/>
    <xsl:if test="tei:rdg">
      <span class="app-marker" title="voir apparat">[<xsl:value-of select="$pos"/>]</span>
    </xsl:if>
  </xsl:template>

  <!-- Entrée d'apparat en bas de page -->
  <xsl:template match="tei:app" mode="apparatus">
    <xsl:variable name="pos">
      <xsl:number count="tei:app" level="any"/>
    </xsl:variable>
    <xsl:if test="tei:rdg">
      <div class="app-entry">
        <span class="app-marker">[<xsl:value-of select="$pos"/>]</span>
        <strong><xsl:value-of select="tei:lem"/></strong>
        <span class="sigle"> <xsl:value-of select="tei:lem/@wit"/></span>
        <xsl:text> ] </xsl:text>
        <xsl:for-each select="tei:rdg">
          <em><xsl:value-of select="."/></em>
          <span class="sigle"> <xsl:value-of select="@wit"/></span>
          <xsl:if test="@type">
            <span class="rdg-type"> (<xsl:value-of select="@type"/>)</span>
          </xsl:if>
          <xsl:if test="position() != last()"><xsl:text> | </xsl:text></xsl:if>
        </xsl:for-each>
      </div>
    </xsl:if>
  </xsl:template>

  <!-- Ignorer le teiHeader -->
  <xsl:template match="tei:teiHeader"/>

</xsl:stylesheet>
"""

# Application de la transformation
xslt_doc = etree.fromstring(xslt_apparat.encode('utf-8'))
transform = etree.XSLT(xslt_doc)
result_html = transform(doc_gen)

print("✓ Transformation XSLT réussie")
print(f"  {len(str(result_html))} caractères générés")

# Affichage dans le notebook
display(HTML(str(result_html)))

### 2.3 — ✏️ Exercice : modifier le XSLT

Modifiez le XSLT pour **afficher le type de variante directement dans le texte** (en info-bulle ou entre parenthèses), plutôt que seulement dans l'apparat.

**Indice** : dans le template `match="tei:app"`, ajoutez un traitement de `@type` sur les `<rdg>`.

> 💡 **Pour les plus avancés** : faites en sorte que les variantes de type `orthographic` soient affichées en gris clair (moins importantes visuellement), et les variantes `lexical` en rouge.

In [ ]:
# ✏️ VOTRE RÉPONSE — copiez et modifiez le XSLT ci-dessus
xslt_modifie = """
<?xml version="1.0" encoding="UTF-8"?>
<xsl:stylesheet version="1.0"
  xmlns:xsl="http://www.w3.org/1999/XSL/Transform"
  xmlns:tei="http://www.tei-c.org/ns/1.0">
  <xsl:output method="html" encoding="UTF-8" indent="yes"/>
  <!-- VOTRE XSLT MODIFIÉ ICI -->
  ...
</xsl:stylesheet>
"""

try:
    xslt_doc2 = etree.fromstring(xslt_modifie.encode('utf-8'))
    transform2 = etree.XSLT(xslt_doc2)
    result2 = transform2(doc_gen)
    display(HTML(str(result2)))
except Exception as e:
    print(f"✗ Erreur : {e}")

---
## Partie 3 — JSON : modélisation et requêtes

### 3.1 — Du TEI au JSON : comprendre la conversion

Observez le parallèle entre le TEI et sa représentation JSON :

In [ ]:
# Représentation JSON de la Genèse 1:1-3
# Comparez avec le TEI de la partie 2 !

edition_json = {
  "titre": "Genesis 1:1-3 — Collation Vetus Latina / Vulgate",
  "langue": "la",
  "temoins": [
    {"sigle": "VL",   "description": "Vetus Latina (Codex Lugdunensis)", "date": "IIe-IVe s."},
    {"sigle": "Vulg", "description": "Vulgata Hieronymi",                "date": "c. 400"},
    {"sigle": "Aug",  "description": "Augustinus, De Genesi ad litteram","date": "c. 401"}
  ],
  "texte": [
    {
      "verset": 1,
      "texte_base": "In principio creavit Deus caelum et terram.",
      "apparat": [
        {
          "lemma": {"texte": "creavit", "temoins": ["Vulg"]},
          "variantes": [
            {"texte": "fecit", "temoins": ["VL", "Aug"], "type": "lexical"}
          ]
        }
      ]
    },
    {
      "verset": 2,
      "texte_base": "Terra autem erat inanis et vacua, et tenebrae erant super faciem abyssi, et spiritus Dei ferebatur super aquas.",
      "apparat": [
        {
          "lemma": {"texte": "abyssi", "temoins": ["Vulg", "VL"]},
          "variantes": [
            {"texte": "abissi", "temoins": ["Aug"], "type": "orthographic"}
          ]
        },
        {
          "lemma": {"texte": "Dei", "temoins": ["Vulg", "Aug"]},
          "variantes": [
            {"texte": "Domini", "temoins": ["VL"], "type": "lexical"}
          ]
        }
      ]
    },
    {
      "verset": 3,
      "texte_base": "Dixitque Deus : fiat lux. Et facta est lux.",
      "apparat": []
    }
  ]
}

# Affichage formaté
print(json.dumps(edition_json, ensure_ascii=False, indent=2))

### 3.2 — Requêtes Python sur la structure JSON

L'intérêt de JSON : on peut l'interroger avec du Python natif, sans XPath.

In [ ]:
# Requête 1 : combien de variantes par type ?
from collections import Counter

types_variantes = []
for verset in edition_json['texte']:
    for app in verset['apparat']:
        for var in app['variantes']:
            types_variantes.append(var['type'])

print("=== Distribution des types de variantes ===")
for t, n in Counter(types_variantes).most_common():
    print(f"  {t:20s} : {n}")

print()

# Requête 2 : quels versets présentent des variantes lexicales ?
print("=== Versets avec variantes lexicales ===")
for verset in edition_json['texte']:
    lex = [v for app in verset['apparat']
             for v in app['variantes']
             if v['type'] == 'lexical']
    if lex:
        print(f"  Verset {verset['verset']} :")
        for v in lex:
            print(f"    '{v['texte']}' — {v['temoins']}")

print()

# Requête 3 : index des témoins et leur nombre de variantes
print("=== Témoins et nombre de variantes portées ===")
temoin_variantes = Counter()
for verset in edition_json['texte']:
    for app in verset['apparat']:
        for var in app['variantes']:
            for t in var['temoins']:
                temoin_variantes[t] += 1

for sigle, info in [(t['sigle'], t['description']) for t in edition_json['temoins']]:
    n = temoin_variantes.get(sigle, 0)
    print(f"  {sigle:6s} ({info[:30]}) : {n} variante(s)")

### 3.3 — ✏️ Exercice : convertir automatiquement le TEI en JSON

Écrivez une fonction Python qui prend un document TEI (comme `doc_gen`) et retourne une structure JSON comparable à `edition_json`.

**Structure attendue :**
```python
{
  "titre": str,
  "langue": str,
  "temoins": [{"sigle": str, "description": str}, ...],
  "texte": [
    {
      "verset": int,
      "apparat": [
        {
          "lemma": {"texte": str, "temoins": [str, ...]},
          "variantes": [{"texte": str, "temoins": [str,...], "type": str}, ...]
        }
      ]
    }
  ]
}
```

**Indices :**
- Utilisez `doc.findall('.//tei:witness', ns)` pour les témoins
- Pour nettoyer les références aux témoins (`#VL #Vulg`) : `wit.replace('#', '').split()`
- `element.text` peut être `None` si l'élément est vide

> 💡 **Pour les plus avancés** : gérez aussi les `<rdg>` vides (type `omission`) et les `<note>` à l'intérieur des `<app>`.

In [ ]:
def tei_to_json(tei_doc):
    """Convertit un document TEI lxml en dictionnaire Python (structure JSON)."""
    ns = {'tei': 'http://www.tei-c.org/ns/1.0',
          'xml': 'http://www.w3.org/XML/1998/namespace'}

    result = {}

    # --- Titre ---
    # VOTRE CODE ICI
    # Indice : tei_doc.find('.//tei:titleStmt/tei:title', ns)
    result['titre'] = ...  # à compléter

    # --- Langue ---
    # Indice : l'attribut xml:lang est sur l'élément <text>
    result['langue'] = ...  # à compléter

    # --- Témoins ---
    # VOTRE CODE ICI
    result['temoins'] = []  # à compléter

    # --- Texte & apparat ---
    result['texte'] = []
    for ab in tei_doc.findall('.//tei:ab', ns):
        verset = {'verset': int(ab.get('n', 0)), 'apparat': []}

        for app in ab.findall('tei:app', ns):
            entree = {}
            lem = app.find('tei:lem', ns)

            # Lemma
            # VOTRE CODE ICI
            entree['lemma'] = ...  # à compléter

            # Variantes
            entree['variantes'] = []
            for rdg in app.findall('tei:rdg', ns):
                # VOTRE CODE ICI
                pass  # à compléter

            verset['apparat'].append(entree)
        result['texte'].append(verset)

    return result


# Test sur le document Genèse
try:
    json_result = tei_to_json(doc_gen)
    print(json.dumps(json_result, ensure_ascii=False, indent=2))
except Exception as e:
    print(f"✗ Erreur : {e}")
    print("  → Complétez les sections marquées 'à compléter'")

<details>
<summary>💡 Solution (cliquez pour révéler)</summary>

```python
def tei_to_json(tei_doc):
    ns = {'tei': 'http://www.tei-c.org/ns/1.0',
          'xml': 'http://www.w3.org/XML/1998/namespace'}
    result = {}

    # Titre
    titre_el = tei_doc.find('.//tei:titleStmt/tei:title', ns)
    result['titre'] = titre_el.text.strip() if titre_el is not None and titre_el.text else ''

    # Langue
    text_el = tei_doc.find('.//tei:text', ns)
    result['langue'] = text_el.get('{http://www.w3.org/XML/1998/namespace}lang', '') if text_el is not None else ''

    # Témoins
    result['temoins'] = []
    for w in tei_doc.findall('.//tei:witness', ns):
        xml_id = w.get('{http://www.w3.org/XML/1998/namespace}id', '')
        result['temoins'].append({'sigle': xml_id, 'description': w.text.strip() if w.text else ''})

    # Texte
    result['texte'] = []
    for ab in tei_doc.findall('.//tei:ab', ns):
        verset = {'verset': int(ab.get('n', 0)), 'apparat': []}
        for app in ab.findall('tei:app', ns):
            entree = {}
            lem = app.find('tei:lem', ns)
            def parse_wit(wit_str):
                return [w.lstrip('#') for w in (wit_str or '').split()]
            entree['lemma'] = {
                'texte': lem.text.strip() if lem is not None and lem.text else '',
                'temoins': parse_wit(lem.get('wit') if lem is not None else '')
            }
            entree['variantes'] = []
            for rdg in app.findall('tei:rdg', ns):
                entree['variantes'].append({
                    'texte': rdg.text.strip() if rdg.text else '',
                    'temoins': parse_wit(rdg.get('wit')),
                    'type': rdg.get('type', '')
                })
            verset['apparat'].append(entree)
        result['texte'].append(verset)
    return result
```
</details>

### 3.4 — ✏️ Exercice final : modéliser un nouveau texte en JSON

Sans TEI cette fois — modélisez **directement en JSON** la tradition manuscrite suivante :

**Virgile, *Énéide* I, 1-4** — trois témoins principaux :

| Sigle | Nom | Date |
|-------|-----|------|
| P | Codex Palatinus (Vat. lat. 1631) | IVe-Ve s. |
| M | Codex Mediceus (Laur. 39.1) | Ve s. |
| R | Codex Romanus (Vat. lat. 3867) | Ve s. |

```
v.1 : Arma virumque cano, Troiae qui primus ab oris
v.2 : Italiam fato profugus Laviniaque venit
v.3 : litora – multum ille et terris iactatus et alto
v.4 : vi superum, saevae memorem Iunonis ob iram
```

**Variantes (simplifiées) :**
- v.1 : `oris` (P, M) vs `horis` (R) — type : orthographic
- v.2 : `Laviniaque` (P, M) vs `Lavinaque` (R) — type : orthographic
- v.4 : `memorem` (P, M, R) — pas de variante ; mais Servius cite `memoris` — type : conjecture

> **Question de réflexion** : comment représentez-vous une conjecture d'un commentateur antique (Servius) qui n'est pas un témoin manuscrit direct ? Comment la TEI le ferait-elle différemment ?

In [ ]:
# ✏️ VOTRE RÉPONSE — construisez le dictionnaire JSON

aeneid_json = {
  "titre": "Aeneis I, 1-4",
  "auteur": "Vergilius",
  "langue": "la",
  "temoins": [
    # COMPLÉTEZ
  ],
  "texte": [
    # COMPLÉTEZ — 4 versets avec apparat
  ]
}

# Validation basique
assert len(aeneid_json['temoins']) == 3, "Il faut 3 témoins"
assert len(aeneid_json['texte']) == 4,   "Il faut 4 versets"

total_var = sum(len(v.get('apparat', [])) for v in aeneid_json['texte'])
print(f"✓ Structure valide : {len(aeneid_json['temoins'])} témoins, {len(aeneid_json['texte'])} versets, {total_var} entrées d'apparat")
print()
print(json.dumps(aeneid_json, ensure_ascii=False, indent=2))

---
## Bilan de séance

| Ce que vous savez faire | Format | Outil |
|-------------------------|--------|-------|
| Encoder un apparat critique | XML TEI | `lxml` |
| Déclarer des témoins dans le `teiHeader` | XML TEI | `lxml` |
| Encoder des phénomènes de transmission (`<unclear>`, `<del>`, `<choice>`) | XML TEI | `lxml` |
| Transformer TEI → HTML avec un XSLT | XSLT 1.0 | `lxml.etree.XSLT` |
| Modéliser une tradition manuscrite | JSON | Python natif |
| Interroger un apparat en JSON | JSON | Python natif |
| Convertir TEI → JSON automatiquement | TEI + JSON | `lxml` + `json` |

### Pour aller plus loin

- **CollateX** (Python) : collation automatique à partir de vos fichiers TEI
- **TEI Publisher** : publication web d'éditions critiques
- **Juxta Commons** : visualisation de la collation
- **`json-schema`** (Python) : valider la structure de vos JSON comme TEI valide votre XML